In [6]:
import pandas as pd
import numpy as np
import re
import csv
import joblib

from scipy.stats import chi2 as chi2_dist

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from xgboost import XGBClassifier

print("Imports OK")

Imports OK


In [7]:
df = pd.read_csv("triage_level.csv")
print("Raw shape:", df.shape)

df = df.dropna(subset=["HPI", "triage"])
print("After dropna:", df.shape)

# ── Parse demographics ──
df["age"]    = df["patient_info"].str.extract(r"Age:\s*(\d+)").astype(float)
df["gender"] = df["patient_info"].str.extract(r"Gender:\s*(\w+)")[0]
df["race"]   = df["patient_info"].str.extract(r"Race:\s*([A-Z\s]+?)(?:,|$)")[0]

# ── Parse vitals ──
df["temp"]  = df["initial_vitals"].str.extract(r"Temperature:\s*([\d.]+)").astype(float)
df["hr"]    = df["initial_vitals"].str.extract(r"Heartrate:\s*([\d.]+)").astype(float)
df["rr"]    = df["initial_vitals"].str.extract(r"resprate:\s*([\d.]+)").astype(float)
df["spo2"]  = df["initial_vitals"].str.extract(r"o2sat:\s*([\d.]+)").astype(float)
df["sbp"]   = df["initial_vitals"].str.extract(r"sbp:\s*([\d.]+)").astype(float)
df["dbp"]   = df["initial_vitals"].str.extract(r"dbp:\s*([\d.]+)").astype(float)

# ── Derived vitals ──
df["map"]            = (df["sbp"] + 2 * df["dbp"]) / 3
df["shock_index"]    = df["hr"] / df["sbp"]
df["pulse_pressure"] = df["sbp"] - df["dbp"]

print("\nMissing counts for key columns:")
print(df[["age","temp","hr","rr","spo2","sbp","dbp"]].isna().sum())
print("\nSample:")
print(df[["age","gender","race","temp","hr","sbp","triage"]].head(3))

Raw shape: (9149, 5)
After dropna: (9148, 5)

Missing counts for key columns:
age       0
temp    581
hr      397
rr      513
spo2    484
sbp     408
dbp     425
dtype: int64

Sample:
    age  gender   race   temp    hr    sbp  triage
0  80.0  Female  WHITE  100.2  95.0   99.0       2
1  69.0    Male  WHITE   97.8  87.0  119.0       2
2  66.0  Female  WHITE   98.7  93.0  166.0       3


In [8]:
def load_clinical_data(path="clinical_data.csv"):
    """Try multiple strategies to load clinical_data.csv robustly."""

    # Strategy 1: pandas Python engine, skip bad lines
    try:
        df_temp = pd.read_csv(
            path,
            engine="python",
            on_bad_lines="skip",
            usecols=["stay_id", "past_medication"],
        )
        print(f"✅ Strategy 1 (pandas python engine): {len(df_temp)} rows")
        return df_temp
    except Exception as e:
        print(f"⚠️ Strategy 1 failed: {type(e).__name__}")

    # Strategy 2: csv module with per-row error handling
    try:
        rows = []
        skipped = 0
        with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
            reader = csv.reader(f)
            header = next(reader)
            stay_idx = header.index("stay_id")
            med_idx  = header.index("past_medication")

            for row in reader:
                try:
                    rows.append({
                        "stay_id": int(row[stay_idx]),
                        "past_medication": row[med_idx] if len(row) > med_idx else None,
                    })
                except (IndexError, ValueError):
                    skipped += 1
                    continue

        df_temp = pd.DataFrame(rows)
        print(f"✅ Strategy 2 (csv module): {len(df_temp)} rows, {skipped} skipped")
        return df_temp
    except Exception as e:
        print(f"⚠️ Strategy 2 failed: {type(e).__name__}: {e}")

    # Strategy 3: read all columns with pandas, skip bad lines, then select
    try:
        full = pd.read_csv(
            path,
            engine="python",
            on_bad_lines="skip",
        )
        df_temp = full[["stay_id", "past_medication"]].copy()
        print(f"✅ Strategy 3 (read all then select): {len(df_temp)} rows")
        return df_temp
    except Exception as e:
        print(f"⚠️ Strategy 3 failed: {type(e).__name__}")

    # Strategy 4: chunked reading — best for very broken files
    try:
        chunks = pd.read_csv(
            path,
            engine="python",
            on_bad_lines="skip",
            chunksize=500,
            usecols=["stay_id", "past_medication"],
        )
        df_temp = pd.concat(chunks, ignore_index=True)
        print(f"✅ Strategy 4 (chunked): {len(df_temp)} rows")
        return df_temp
    except Exception as e:
        print(f"❌ All strategies failed: {e}")
        raise


med_raw = load_clinical_data()
print(f"\nLoaded {len(med_raw)} records")
print(med_raw.head(3))

✅ Strategy 1 (pandas python engine): 9149 rows

Loaded 9149 records
    stay_id                                    past_medication
0  32822973  \nmedications at ___:\nfluticasone 50 mcg nasa...
1  31591237  \nprednisone 40'\nalendronate 70'\nazathioprin...
2  36819102  \nthe preadmission medication list is accurate...


In [9]:
def count_meds(s):
    """Count medications listed in past_medication."""
    if pd.isna(s) or not isinstance(s, str):
        return 0
    parts = re.split(r"[;\n,]", s)
    return len([p for p in parts if p.strip()])

med_df = med_raw.copy()
med_df["num_meds"] = med_df["past_medication"].apply(count_meds)

txt = med_df["past_medication"].fillna("").str.lower()

# Drug class flags
med_df["has_insulin"]    = txt.str.contains(r"insulin", na=False).astype(int)
med_df["has_anticoag"]   = txt.str.contains(
    r"heparin|warfarin|apixaban|rivaroxaban|enoxaparin|dabigatran", na=False
).astype(int)
med_df["has_statin"]     = txt.str.contains(
    r"atorvastatin|rosuvastatin|simvastatin|pravastatin|lovastatin", na=False
).astype(int)
med_df["has_bp_med"]     = txt.str.contains(
    r"lisinopril|losartan|metoprolol|amlodipine|atenolol|hydrochlorothiazide|"
    r"carvedilol|valsartan|olmesartan|propranolol", na=False
).astype(int)
med_df["has_steroid"]    = txt.str.contains(
    r"prednisone|methylprednisolone|hydrocortisone|dexamethasone", na=False
).astype(int)
med_df["has_inhaler"]    = txt.str.contains(
    r"albuterol|fluticasone|ipratropium|budesonide|salmeterol|tiotropium", na=False
).astype(int)
med_df["has_opioid"]     = txt.str.contains(
    r"oxycodone|hydrocodone|morphine|fentanyl|tramadol|codeine", na=False
).astype(int)
med_df["has_diuretic"]   = txt.str.contains(
    r"furosemide|bumetanide|spironolactone|torsemide", na=False
).astype(int)

MED_FEATURES = [
    "num_meds", "has_insulin", "has_anticoag", "has_statin",
    "has_bp_med", "has_steroid", "has_inhaler", "has_opioid", "has_diuretic",
]

print("Med features distribution:")
print(med_df[MED_FEATURES].describe().round(3))
print(f"\nRows with at least 1 med: {(med_df['num_meds'] > 0).sum()}")

Med features distribution:
       num_meds  has_insulin  has_anticoag  has_statin  has_bp_med  \
count  9149.000     9149.000      9149.000    9149.000    9149.000   
mean      8.036        0.053         0.091       0.273       0.390   
std       9.298        0.223         0.287       0.445       0.488   
min       0.000        0.000         0.000       0.000       0.000   
25%       2.000        0.000         0.000       0.000       0.000   
50%       5.000        0.000         0.000       0.000       0.000   
75%      12.000        0.000         0.000       1.000       1.000   
max     135.000        1.000         1.000       1.000       1.000   

       has_steroid  has_inhaler  has_opioid  has_diuretic  
count     9149.000     9149.000    9149.000      9149.000  
mean         0.060        0.153       0.153         0.103  
std          0.238        0.360       0.360         0.304  
min          0.000        0.000       0.000         0.000  
25%          0.000        0.000       0.00

In [10]:
df = df.merge(
    med_df[["stay_id"] + MED_FEATURES],
    on="stay_id",
    how="left",
)

# Fill missing: if no record in clinical_data, assume 0 meds
df[MED_FEATURES] = df[MED_FEATURES].fillna(0)

print(f"Final shape: {df.shape}")
print(f"Rows with med features: {(df['num_meds'] > 0).sum()} / {len(df)}")
print(f"Rows without med record (filled with 0): {(df['num_meds'] == 0).sum()}")

Final shape: (9148, 26)
Rows with med features: 8628 / 9148
Rows without med record (filled with 0): 520


In [11]:
# CRITICAL: same random_state=42 and stratify=y as Demo 1 baseline
indices = df.index.to_numpy()
y       = df["triage"].to_numpy()

idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices, y, test_size=0.30, random_state=42, stratify=y
)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

train_df = df.loc[idx_train].reset_index(drop=True)
val_df   = df.loc[idx_val].reset_index(drop=True)
test_df  = df.loc[idx_test].reset_index(drop=True)

y_train = train_df["triage"].to_numpy()
y_val   = val_df["triage"].to_numpy()
y_test  = test_df["triage"].to_numpy()

# XGBoost needs 0-indexed labels
LABEL_MAP = {1: 0, 2: 1, 3: 2, 4: 3}
INV_MAP   = {0: 1, 1: 2, 2: 3, 3: 4}

y_train_xgb = np.array([LABEL_MAP[v] for v in y_train])
y_val_xgb   = np.array([LABEL_MAP[v] for v in y_val])
y_test_xgb  = np.array([LABEL_MAP[v] for v in y_test])

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Train class distribution: {pd.Series(y_train).value_counts().sort_index().to_dict()}")

Train: 6403 | Val: 1372 | Test: 1373
Train class distribution: {1: 614, 2: 2309, 3: 3435, 4: 45}


In [12]:
TEXT_COL   = "HPI"

DEMO_NUM   = ["age"]
DEMO_CAT   = ["gender", "race"]

HIST_NUM   = MED_FEATURES  # from Cell 4

VITALS_NUM = [
    "temp", "hr", "rr", "spo2", "sbp", "dbp",
    "map", "shock_index", "pulse_pressure",
]

print(f"Demographics: {len(DEMO_NUM)} numeric + {len(DEMO_CAT)} categorical")
print(f"History (meds): {len(HIST_NUM)}")
print(f"Vitals: {len(VITALS_NUM)}")

Demographics: 1 numeric + 2 categorical
History (meds): 9
Vitals: 9


In [13]:
def make_pipeline(num_cols, cat_cols):
    transformers = [
        ("text", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words="english",
            token_pattern=r"\b[a-zA-Z]{2,}\b",
            min_df=2,
            max_df=0.95,
        ), TEXT_COL),
    ]

    if num_cols:
        transformers.append(("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), num_cols))

    if cat_cols:
        transformers.append(("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols))

    return Pipeline([
        ("prep", ColumnTransformer(transformers)),
        ("clf", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            eval_metric="mlogloss",
            n_jobs=-1,
        )),
    ])

print("Pipeline factory ready")

Pipeline factory ready


In [14]:
def train_and_eval(name, num_cols, cat_cols):
    cols = [TEXT_COL] + num_cols + cat_cols
    pipe = make_pipeline(num_cols, cat_cols)
    pipe.fit(train_df[cols], y_train_xgb)

    val_pred_xgb  = pipe.predict(val_df[cols])
    test_pred_xgb = pipe.predict(test_df[cols])

    val_pred  = np.array([INV_MAP[v] for v in val_pred_xgb])
    test_pred = np.array([INV_MAP[v] for v in test_pred_xgb])

    result = {
        "name": name,
        "val_acc":  accuracy_score(y_val,  val_pred),
        "val_f1":   f1_score(y_val,  val_pred, average="macro"),
        "test_acc": accuracy_score(y_test, test_pred),
        "test_f1":  f1_score(y_test, test_pred, average="macro"),
        "test_f1_weighted": f1_score(y_test, test_pred, average="weighted"),
        "test_pred": test_pred,
        "pipe": pipe,
    }

    print(f"{name:30s} | Val Acc {result['val_acc']:.4f} | "
          f"Test Acc {result['test_acc']:.4f} | "
          f"Test Macro F1 {result['test_f1']:.4f}")
    return result

print("Helper ready")

Helper ready


In [15]:
m1 = train_and_eval("M1: HPI only", [], [])

M1: HPI only                   | Val Acc 0.6385 | Test Acc 0.6358 | Test Macro F1 0.3690


In [16]:
m2 = train_and_eval("M2: + demographics", DEMO_NUM, DEMO_CAT)

M2: + demographics             | Val Acc 0.6348 | Test Acc 0.6577 | Test Macro F1 0.3955


In [17]:
m3 = train_and_eval(
    "M3: + medical history",
    DEMO_NUM + HIST_NUM,
    DEMO_CAT,
)

M3: + medical history          | Val Acc 0.6385 | Test Acc 0.6511 | Test Macro F1 0.3928


In [18]:
m4 = train_and_eval(
    "M4: + vitals (full)",
    DEMO_NUM + HIST_NUM + VITALS_NUM,
    DEMO_CAT,
)

M4: + vitals (full)            | Val Acc 0.6880 | Test Acc 0.6883 | Test Macro F1 0.5001


In [19]:
all_results = [m1, m2, m3, m4]

comparison = pd.DataFrame([
    {
        "Model":             r["name"],
        "Val Acc":           f"{r['val_acc']:.4f}",
        "Test Acc":          f"{r['test_acc']:.4f}",
        "Test Macro F1":     f"{r['test_f1']:.4f}",
        "Test Weighted F1":  f"{r['test_f1_weighted']:.4f}",
        "Δ Acc vs M1":       f"{r['test_acc'] - m1['test_acc']:+.4f}",
        "Δ Macro F1 vs M1":  f"{r['test_f1'] - m1['test_f1']:+.4f}",
    }
    for r in all_results
])

print("=" * 110)
print("PERSONALIZATION COMPARISON (Demo 2)")
print("=" * 110)
print(comparison.to_string(index=False))

PERSONALIZATION COMPARISON (Demo 2)
                Model Val Acc Test Acc Test Macro F1 Test Weighted F1 Δ Acc vs M1 Δ Macro F1 vs M1
         M1: HPI only  0.6385   0.6358        0.3690           0.6131     +0.0000          +0.0000
   M2: + demographics  0.6348   0.6577        0.3955           0.6372     +0.0218          +0.0266
M3: + medical history  0.6385   0.6511        0.3928           0.6314     +0.0153          +0.0238
  M4: + vitals (full)  0.6880   0.6883        0.5001           0.6812     +0.0524          +0.1311


In [20]:
LABELS = [1, 2, 3, 4]
LABEL_NAMES = ["ESI 1", "ESI 2", "ESI 3", "ESI 4"]

for r in all_results:
    print(f"\n{'=' * 60}")
    print(f"{r['name']}")
    print(f"{'=' * 60}")
    print(classification_report(
        y_test, r["test_pred"],
        labels=LABELS,
        target_names=LABEL_NAMES,
        zero_division=0,
    ))


M1: HPI only
              precision    recall  f1-score   support

       ESI 1       0.42      0.12      0.19       132
       ESI 2       0.55      0.54      0.54       495
       ESI 3       0.69      0.80      0.75       736
       ESI 4       0.00      0.00      0.00        10

    accuracy                           0.64      1373
   macro avg       0.42      0.37      0.37      1373
weighted avg       0.61      0.64      0.61      1373


M2: + demographics
              precision    recall  f1-score   support

       ESI 1       0.52      0.17      0.25       132
       ESI 2       0.58      0.56      0.57       495
       ESI 3       0.71      0.82      0.76       736
       ESI 4       0.00      0.00      0.00        10

    accuracy                           0.66      1373
   macro avg       0.45      0.39      0.40      1373
weighted avg       0.64      0.66      0.64      1373


M3: + medical history
              precision    recall  f1-score   support

       ESI 1      

In [21]:
def mcnemar_test(pred_a, pred_b, y_true):
    """McNemar's test between two paired predictors."""
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)

    a_only = int(np.sum(correct_a & ~correct_b))
    b_only = int(np.sum(~correct_a & correct_b))

    if (a_only + b_only) == 0:
        return np.nan, np.nan

    chi2 = (abs(a_only - b_only) - 1) ** 2 / (a_only + b_only)
    p_value = 1 - chi2_dist.cdf(chi2, df=1)
    return chi2, p_value


print("=" * 80)
print("McNemar's Test vs Baseline (M1)")
print("=" * 80)
print(f"{'Comparison':40s} {'chi²':>8s} {'p-value':>10s} {'Result':>15s}")
print("-" * 80)

for r in [m2, m3, m4]:
    chi2, p = mcnemar_test(m1["test_pred"], r["test_pred"], y_test)
    sig = "SIGNIFICANT" if (not np.isnan(p) and p < 0.05) else "not significant"
    print(f"M1 vs {r['name']:32s} {chi2:8.3f} {p:10.4f} {sig:>15s}")

McNemar's Test vs Baseline (M1)
Comparison                                   chi²    p-value          Result
--------------------------------------------------------------------------------
M1 vs M2: + demographics                  8.947     0.0028     SIGNIFICANT
M1 vs M3: + medical history               4.211     0.0402     SIGNIFICANT
M1 vs M4: + vitals (full)                26.532     0.0000     SIGNIFICANT


In [22]:
for r in all_results:
    cm = confusion_matrix(y_test, r["test_pred"], labels=LABELS)
    print(f"\n{r['name']} — Confusion Matrix")
    print(pd.DataFrame(
        cm,
        index=["Actual 1", "Actual 2", "Actual 3", "Actual 4"],
        columns=["Pred 1", "Pred 2", "Pred 3", "Pred 4"],
    ))


M1: HPI only — Confusion Matrix
          Pred 1  Pred 2  Pred 3  Pred 4
Actual 1      16      77      39       0
Actual 2      16     265     214       0
Actual 3       6     138     592       0
Actual 4       0       2       8       0

M2: + demographics — Confusion Matrix
          Pred 1  Pred 2  Pred 3  Pred 4
Actual 1      22      76      34       0
Actual 2      14     277     204       0
Actual 3       6     126     604       0
Actual 4       0       2       8       0

M3: + medical history — Confusion Matrix
          Pred 1  Pred 2  Pred 3  Pred 4
Actual 1      22      78      32       0
Actual 2      14     276     205       0
Actual 3       4     136     596       0
Actual 4       0       1       9       0

M4: + vitals (full) — Confusion Matrix
          Pred 1  Pred 2  Pred 3  Pred 4
Actual 1      72      42      18       0
Actual 2      13     276     206       0
Actual 3       2     137     597       0
Actual 4       0       1       9       0


In [23]:
import os
os.makedirs("models", exist_ok=True)

saved_files = []
for r in all_results:
    safe_name = r["name"].split(":")[0].strip().lower().replace(" ", "_")
    fname = f"models/{safe_name}.pkl"
    joblib.dump(r["pipe"], fname)
    saved_files.append(fname)
    print(f"Saved {fname}")

# Download all models to your local Mac
from google.colab import files
for f in saved_files:
    files.download(f)

Saved models/m1.pkl
Saved models/m2.pkl
Saved models/m3.pkl
Saved models/m4.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>